In [ ]:
!pip install keras
!pip install scikeras
!pip install tensorflow

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.model_selection import GridSearchCV, KFold
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, LSTM, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasClassifier

In [ ]:
# Specify the path to your Excel file
file_path = '/content/Data Fix.xlsx'

# Read the Excel file
df = pd.read_excel(file_path)

# Display the first few rows of the DataFrame
print(df.head())


                                                teks  polarity
0  apa apa company lokal asal wfa mungkin bukan k...         0
1        sorry baru liat notif jakarta wfa permanent         0
2                 yay nambah opsi baru wfa spot cibi         0
3  jadi laksana proyek susah kalau wfa asli cuti ...         0
4  tuju karir sekarang fokus lebih work life bala...         1


In [ ]:
# Definisi Variabel
X = df['teks'] # Fitur: review
y = df['polarity'] # Label: negatif (0) atau positif (1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Menghitung frekuensi label 0 dan 1 pada data pelatihan
train_freq = y_train.value_counts()

# Menghitung frekuensi label 0 dan 1 pada data pengujian
test_freq = y_test.value_counts()

# Menampilkan hasil
print("Frekuensi Label pada Data Pelatihan:")
print(train_freq)
print("\nFrekuensi Label pada Data Pengujian:")
print(test_freq)

Frekuensi Label pada Data Pelatihan:
polarity
0    6287
1    4741
Name: count, dtype: int64

Frekuensi Label pada Data Pengujian:
polarity
0    1605
1    1152
Name: count, dtype: int64


In [ ]:
# Assuming X_train and X_test are pandas DataFrames or Series containing the text data
# Check for missing values
print(X_train.isnull().sum())
print(X_test.isnull().sum())

# Option 2: Replace NaNs with Empty Strings
X_train = X_train.fillna("")
X_test = X_test.fillna("")

# Ekstraksi fitur menggunakan TF-IDF
vectorizer = TfidfVectorizer(max_features=5000) # Maksimal 5000 fitur
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)


1
0
(11028, 5000)
(2757, 5000)


In [ ]:
# Transform data to required format for CNN-LSTM
X_train_tfidf_array = X_train_tfidf.toarray().reshape(X_train_tfidf.shape[0], X_train_tfidf.shape[1], 1)
X_test_tfidf_array = X_test_tfidf.toarray().reshape(X_test_tfidf.shape[0], X_test_tfidf.shape[1], 1)

In [ ]:
# Import necessary libraries
from keras.models import Sequential
from keras.layers import Embedding, Dropout, Conv1D, MaxPooling1D, LSTM, Dense, Activation

# Define necessary parameters (make sure to set appropriate values)
max_features = 20000  # Vocabulary size
embedding_size = 128  # Embedding dimension
maxlen = 100  # Maximum length of sequences
filters = 64  # Number of filters in Conv1D
kernel_size = 3  # Kernel size in Conv1D
pool_size = 2  # Pool size in MaxPooling1D
lstm_output_size = 70  # Output size of LSTM
batch_size = 32  # Batch size
epochs = 10  # Number of epochs

# Building the model
model = Sequential()
model.add(Embedding(input_dim=max_features, output_dim=embedding_size, input_length=maxlen))
model.add(Dropout(0.25))
model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='valid', activation='relu', strides=1))
model.add(MaxPooling1D(pool_size=pool_size))
model.add(LSTM(units=lstm_output_size))
model.add(Dense(1))
model.add(Activation('sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()


Model: "sequential_162"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 100, 128)          2560000   
                                                                 
 dropout_1 (Dropout)         (None, 100, 128)          0         
                                                                 
 conv1d_1 (Conv1D)           (None, 98, 64)            24640     
                                                                 
 max_pooling1d_1 (MaxPoolin  (None, 49, 64)            0         
 g1D)                                                            
                                                                 
 lstm_1 (LSTM)               (None, 70)                37800     
                                                                 
 dense_1 (Dense)             (None, 1)                 71        
                                                    

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

# Ensure the input data is in the correct dense format
if isinstance(X_train_tfidf, csr_matrix):
    X_train_tfidf = X_train_tfidf.toarray()

if isinstance(X_test_tfidf, csr_matrix):
    X_test_tfidf = X_test_tfidf.toarray()

# Import necessary libraries
from keras.models import Sequential
from keras.layers import Dense, Dropout

# Define necessary parameters
input_dim = X_train_tfidf.shape[1]  # This should match the dimensionality of your TF-IDF vectors
batch_size = 32
epochs = 10

# Building the model for TF-IDF input
model = Sequential()
model.add(Dense(512, input_shape=(input_dim,), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train_tfidf, y_train, batch_size=batch_size, epochs=epochs, validation_data=(X_test_tfidf, y_test))

# Evaluate the model
score, acc = model.evaluate(X_test_tfidf, y_test, batch_size=batch_size)

# Print test score and accuracy
print('Test score:', score)
print('Test accuracy:', acc)


Epoch 1/10
345/345 [==============================] - 42s 116ms/step - loss: 0.4530 - accuracy: 0.7670 - val_loss: 0.2976 - val_accuracy: 0.8643
Epoch 2/10
345/345 [==============================] - 39s 112ms/step - loss: 0.1870 - accuracy: 0.9242 - val_loss: 0.2796 - val_accuracy: 0.8799
Epoch 3/10
345/345 [==============================] - 36s 104ms/step - loss: 0.0845 - accuracy: 0.9703 - val_loss: 0.3180 - val_accuracy: 0.8839
Epoch 4/10
345/345 [==============================] - 36s 105ms/step - loss: 0.0314 - accuracy: 0.9924 - val_loss: 0.3828 - val_accuracy: 0.8905
Epoch 5/10
345/345 [==============================] - 37s 108ms/step - loss: 0.0158 - accuracy: 0.9963 - val_loss: 0.4300 - val_accuracy: 0.8868
Epoch 6/10
345/345 [==============================] - 36s 103ms/step - loss: 0.0122 - accuracy: 0.9962 - val_loss: 0.4578 - val_accuracy: 0.8883
Epoch 7/10
345/345 [==============================] - 35s 102ms/step - loss: 0.0078 - accuracy: 0.9984 - val_loss: 0.4790 - val_ac

In [ ]:
import matplotlib.pyplot as plt

# Data from training
epochs = list(range(1, 11))
accuracy = [0.7670, 0.9242, 0.9703, 0.9924, 0.9963, 0.9962, 0.9984, 0.9978, 0.9992, 0.9990]
val_accuracy = [0.8643, 0.8799, 0.8839, 0.8905, 0.8868, 0.8883, 0.8886, 0.8839, 0.8905, 0.8796]

# Plotting the accuracies
plt.figure(figsize=(10, 6))
plt.plot(epochs, accuracy, label='Training Accuracy', marker='o')
plt.plot(epochs, val_accuracy, label='Validation Accuracy', marker='o')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy over Epochs')
plt.legend()
plt.grid(True)
plt.xticks(epochs)
plt.show()


In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
y_pred_prob = model.predict(X_test_tfidf)
y_pred = (y_pred_prob > 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')



87/87 [==============================] - 3s 32ms/step
Accuracy: 0.8795792528110264
Precision: 0.900390625
Recall: 0.8003472222222222
F1 Score: 0.8474264705882353


In [ ]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.87      0.94      0.90      1605
           1       0.90      0.80      0.85      1152

    accuracy                           0.88      2757
   macro avg       0.88      0.87      0.87      2757
weighted avg       0.88      0.88      0.88      2757



In [ ]:
import numpy as np

# Hitung jumlah masing-masing kelas dalam y_pred
count_class_0 = np.sum(y_pred == 0)
count_class_1 = np.sum(y_pred == 1)

print(f'Jumlah kelas 0 (negatif): {count_class_0}')
print(f'Jumlah kelas 1 (positif): {count_class_1}')


Jumlah kelas 0 (negatif): 1733
Jumlah kelas 1 (positif): 1024
